In [1]:
!pip install xmltodict

## Notebook para gerar um conjunto de dados para treinar os modelos baseados em llama

In [1]:
from odf.opendocument import OpenDocumentText
from odf.style import (Style, TextProperties, ParagraphProperties,
                       ListLevelProperties, TabStop, TabStops)
from odf.text import (H, P, List, ListItem, ListStyle, ListLevelStyleNumber,
                      ListLevelStyleBullet)
from odf import teletype

from odf.opendocument import OpenDocumentText, OpenDocumentDrawing
from odf.draw import Frame, Image, Page
from odf.style import PageLayout, MasterPage, Header, Footer
from odf.text import P
from odf import table
from datetime import datetime
from odf.style import Style, GraphicProperties

from odf.style import Style, MasterPage, PageLayout, PageLayoutProperties, \
TextProperties, GraphicProperties, ParagraphProperties, DrawingPageProperties


from odf.opendocument import load
import xmltodict
import json

import pandas as pd
import glob

In [2]:
def process_number(text, padding = 3, max_decimal = 5):
    text = text.replace("cm","")
    flo = float(text) + padding
    return str(flo)[:max_decimal]+"cm"

def extract_json_odt(path_odt):
    """
    """
    odt = load(path_odt)
    dict_odt = xmltodict.parse(odt.contentxml())
    body = dict_odt['office:document-content']['office:body']['office:text']['text:p']
    
    list_output = []
    
    count_texts = 0
    for item in body:
        if '#text' in item:
            count_texts += 1
    
    for item in body:
        # for texts
        if '#text' in item:
            list_output.append({"text": item['#text']})
        # for images
        elif 'draw:frame' in item:
            if not(type(item['draw:frame']) is list):
                item['draw:frame'] = [item['draw:frame']]
            for fig in item['draw:frame']:
                dict_image = {
                    "image": fig['draw:image']['@xlink:href'].split("/")[-1],
                    "width": fig['@svg:width'],
                    "height": fig['@svg:height'],
                    "x": process_number(fig['@svg:x'], padding = 3),
                    "y": process_number(fig['@svg:y'], padding = max(4,count_texts * 1.5)),
                }
                list_output.append(dict_image)
            
    
    return list_output

def construct_odt_from_json(json, path_images, path_save):
    """
    """
    textdoc = OpenDocumentText()
    
    frstyle = Style(name = 'frstyle', parentstylename="Graphics", family="graphic")
    frstyle.addElement(GraphicProperties(verticalpos="from-top", verticalrel="page", horizontalpos="from-left", horizontalrel="page"))
    textdoc.automaticstyles.addElement(frstyle)

    for item in json:
        if not('text' in item):
            photoframe = Frame(width =  item['width'], 
                               height = item['height'], 
                               x = item['x'], 
                               y = item['y'],
                               anchortype="page",
                               stylename = frstyle
            )
            href = textdoc.addPicture(path_images + item['image'])
            photoframe.addElement(Image(href=href))
            textdoc.text.addElement(photoframe)
    
    for item in json:
        if 'text' in item:
            p = P()
            paragraph_text = item['text']
            teletype.addTextToElement(p, paragraph_text)
            textdoc.text.addElement(p)
            
    textdoc.save(path_save)

In [4]:
skelleton = extract_json_odt("hand_made_odt_dataset/matematica/adição/example4.odt")
construct_odt_from_json(skelleton, "hand_made_odt_dataset/images/", "hand_made_odt_dataset/aux_reconstructed.odt")

## Fazendo o conjunto de dados para treino do llm

In [8]:
path_csv_images = "hand_made_odt_dataset/images/0descriptions.csv"
try:
    desc_images = pd.read_csv(path_csv_images)
except FileNotFoundError:
    desc_images = pd.DataFrame([],columns = ["arq","description"])
    desc_images.to_csv(path_csv_images,index=False)

## Imagens que estão sem descrição

In [9]:
for arq in glob.glob("hand_made_odt_dataset/images/**.png"):
    name = arq.split('/')[-1]
    if not(name in list(desc_images['arq'])):
        print(name)

In [15]:
def generate_question(path_odt, materia, assunto, path_description):
    
    prompt = \
f"""Gere uma questão de {materia} em formato JSON que ajude crianças do 
ensino fundamental os principios da {assunto} de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

"""
    
    descr = pd.read_csv(path_description)
    for index,line in descr.iterrows():
        prompt += f"{line['arq']} :: {line['description']}\n"
        
    example = path_odt.split("/")[-1]
#     prompt += example
    
    
    skelleton = json.dumps(extract_json_odt(path_odt), ensure_ascii = False)
    
    with open(path_odt.replace(".odt",".txt"),"r") as file:
        description_question = file.read()
    
    text = f"<s>[INST] {prompt}\n[/INST]\n{str(skelleton)}\n\n{description_question}</s>"
    
    return [example, prompt,skelleton, text]

def generate_questions(path_odts,path_description):
    
    lines = []
    for odt in glob.iglob(f"{path_odts}/*.odt", recursive = True):
        _, materia, assunto, _ = odt.split('/')
        lines.append(
            generate_question(odt, materia, assunto, path_description)
        )
    
    return pd.DataFrame(lines, columns = ['file','prompt','response', 'text'])

In [16]:
dataset = generate_questions(
    "hand_made_odt_dataset/**",
    "hand_made_odt_dataset/images/0descriptions.csv"
)
dataset.to_csv("dataset_prompts/dataset.csv",index=False)
dataset

,file,prompt,response,text
0,example8.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Maria tem 5 pirulitos, e a quantid...",<s>[INST] Gere uma questão de matematica em fo...
1,example5.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Teresa tem 7 pirulitos e deu 3 para...",<s>[INST] Gere uma questão de matematica em fo...
2,example7.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Qual a soma dos números abaixo ?""}...",<s>[INST] Gere uma questão de matematica em fo...
3,example2.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Faça a dos conjuntos e ligue com o...",<s>[INST] Gere uma questão de matematica em fo...
4,example9.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Subtraia os valores dos lados dos d...",<s>[INST] Gere uma questão de matematica em fo...
5,example4.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Observe as figuras e responda as se...",<s>[INST] Gere uma questão de matematica em fo...
6,example1.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Faça a dos conjuntos e escreva o r...",<s>[INST] Gere uma questão de matematica em fo...
7,example3.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Escreva os números que faltam para ...",<s>[INST] Gere uma questão de matematica em fo...
8,example10.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Em um cacho de uvas tem 9 uvas, se ...",<s>[INST] Gere uma questão de matematica em fo...
9,example6.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Márcio tem 5 balões de são joão, se...",<s>[INST] Gere uma questão de matematica em fo...


In [17]:
print(dataset['text'][5])

<s>[INST] Gere uma questão de matematica em formato JSON que ajude crianças do 
ensino fundamental os principios da subtração de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

numero_6_preto_branco.png :: figura do número 6 em preto e branco
numero_9_preto_branco.png :: figura do número 9 em preto e branco
boneca.png :: imagem de uma boneca preto e branco
numero_8_preto_branco.png :: figura do número 8 em preto e branco
numero_5_preto_branco.png :: figura do número 5 em preto e branco
abacaxi_preto_branco.png :: figura de um abacaxi em preto e branco
numero_4_preto_branco.png :: figura do número 4 em preto e branco
sinal_mais.png :: imagem de um sinal de mais
dado_5.png :: imagem de um dado mostrando o lado com número 5
carrinho.png :: imagem de um carro de brinquedo em preto e branco
dado_2.png :: imagem de um dado mostrando o lado com número 2
numero_1_preto_branco.png :: figura do número 1 em preto e branco
numero_2_preto_branco.png :: figura do número 2 em p

## Output from the model

In [4]:
output = """[{"text": "João tem 5 bonecas e recebe mais de 7 bonecas, faça a adição:"}, {"image": "boneca.png", "width": "2.332cm", "height": "2.456cm", "x": "4.337cm", "y": "5.423cm"}, {"image": "boneca.png", "width": "2.265cm", "height": "2.397cm", "x": "8.975cm", "y": "6.556cm"}, {"image": "boneca.png", "width": "2.227cm", "height": "2.478cm", "x": "11.09cm", "y": "4.246cm"}, {"image": "boneca.png", "width": "2.144cm", "height": "2.286cm", "x": "11.65cm", "y": "7.097cm"}, {"image": "boneca.png", "width": "2.059cm", "height": "2.227cm", "x": "5.72cm", "y": "7.803cm"}]"""
skelleton = json.loads(output)
construct_odt_from_json(skelleton, "hand_made_odt_dataset/images/", "reconstructed/aux_reconstructed.odt")